# Isotools alternative splicing analysis (per sample)

## Setup

In [1]:
from isotools import Transcriptome
import pandas as pd
import numpy as np
import os

# Define paths
path = '/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output'
outpath = f'{path}/01.AS'

# Define filtering parameters
params = {'min_total': 16, 'min_alt_fraction': 0.125}

## Individual samples

In [2]:
print("\n" + "="*80)
print("PART 1: ANALYZING INDIVIDUAL SAMPLES")
print("="*80)

# Load the individual samples transcriptome
isoseq_individual = Transcriptome.load(f'{path}/A_najas_isotools_filtered.pkl')

# Define individual samples
individual_samples = ['I1', 'I2', 'I3', 'I4', 'I5F', 'I5M', 'Fad', 'Mad']

# Store results for individual samples
results_individual = {}
problematic_genes_individual = {}

for sample in individual_samples:
    print(f"\n{'='*60}")
    print(f"Processing sample: {sample}")
    print(f"Parameters: min_total={params['min_total']}, min_alt_fraction={params['min_alt_fraction']}")
    print(f"{'='*60}")
    
    # First pass: identify problematic genes
    problematic_genes = []
    total_genes = 0
    
    for gene in isoseq_individual.iter_genes():
        total_genes += 1
        try:
            events = isoseq_individual.alternative_splicing_events(
                region=f"{gene.chrom}:{gene.start}-{gene.end}", 
                min_total=params['min_total'], 
                min_alt_fraction=params['min_alt_fraction'],
                samples=[sample]
            )
        except Exception:
            problematic_genes.append(gene.id)
    
    print(f"  Found {len(problematic_genes)} problematic genes out of {total_genes}")
    
    # Second pass: collect events
    all_events = []
    genes_with_events = 0
    
    for gene in isoseq_individual.iter_genes():
        if gene.id in problematic_genes:
            continue
        try:
            events = isoseq_individual.alternative_splicing_events(
                region=f"{gene.chrom}:{gene.start}-{gene.end}", 
                min_total=params['min_total'], 
                min_alt_fraction=params['min_alt_fraction'],
                samples=[sample]
            )
            if events is not None and len(events) > 0:
                all_events.append(events)
                genes_with_events += 1
        except Exception:
            problematic_genes.append(gene.id)
    
    # Combine events
    if all_events:
        splice_events = pd.concat(all_events, ignore_index=True)
        results_individual[sample] = splice_events
        problematic_genes_individual[sample] = problematic_genes
        print(f"  Found {len(splice_events)} events across {genes_with_events} genes")
        
        # Save individual results
        output_dir = f'{outpath}/individual/{sample}'
        os.makedirs(output_dir, exist_ok=True)
        splice_events.to_csv(f'{output_dir}/{sample}_splicing_events.csv', index=False)
        
        with open(f'{output_dir}/{sample}_problematic_genes.txt', 'w') as f:
            for gene_id in problematic_genes:
                f.write(f"{gene_id}\n")
    else:
        print(f"  No events found for {sample}")
        results_individual[sample] = pd.DataFrame()


PART 1: ANALYZING INDIVIDUAL SAMPLES

Processing sample: I1
Parameters: min_total=16, min_alt_fraction=0.125
  Found 1709 problematic genes out of 15472
  Found 32704 events across 4324 genes

Processing sample: I2
Parameters: min_total=16, min_alt_fraction=0.125
  Found 1728 problematic genes out of 15472
  Found 33797 events across 4252 genes

Processing sample: I3
Parameters: min_total=16, min_alt_fraction=0.125
  Found 1696 problematic genes out of 15472
  Found 30385 events across 3993 genes

Processing sample: I4
Parameters: min_total=16, min_alt_fraction=0.125
  Found 1742 problematic genes out of 15472
  Found 27719 events across 4106 genes

Processing sample: I5F
Parameters: min_total=16, min_alt_fraction=0.125
  Found 1702 problematic genes out of 15472
  Found 22734 events across 3647 genes

Processing sample: I5M
Parameters: min_total=16, min_alt_fraction=0.125
  Found 1758 problematic genes out of 15472
  Found 27461 events across 4212 genes

Processing sample: Fad
Parame

## Pooled groups

In [3]:
print("\n" + "="*80)
print("PART 2: ANALYZING POOLED GROUPS")
print("="*80)

# Load the pooled transcriptome (different pickle file)
isoseq_pooled = Transcriptome.load(f'{path}/A_najas_isotools_filtered_pooled.pkl')

# Define groups and their constituent samples
groups = {
    'I5': ['I5F', 'I5M'],
    'AD': ['Fad', 'Mad']
}

# Store results for pooled groups
results_pooled = {}
problematic_genes_pooled = {}

for group_name, sample_list in groups.items():
    print(f"\n{'='*60}")
    print(f"Processing group: {group_name}")
    print(f"Samples in group: {sample_list}")
    print(f"Parameters: min_total={params['min_total']}, min_alt_fraction={params['min_alt_fraction']}")
    print(f"{'='*60}")
    
    # First pass: identify problematic genes
    problematic_genes = []
    total_genes = 0
    
    for gene in isoseq_pooled.iter_genes():
        total_genes += 1
        try:
            events = isoseq_pooled.alternative_splicing_events(
                region=f"{gene.chrom}:{gene.start}-{gene.end}", 
                min_total=params['min_total'], 
                min_alt_fraction=params['min_alt_fraction'],
                samples=sample_list
            )
        except Exception:
            problematic_genes.append(gene.id)
    
    print(f"  Found {len(problematic_genes)} problematic genes out of {total_genes}")
    
    # Second pass: collect events
    all_events = []
    genes_with_events = 0
    
    for gene in isoseq_pooled.iter_genes():
        if gene.id in problematic_genes:
            continue
        try:
            events = isoseq_pooled.alternative_splicing_events(
                region=f"{gene.chrom}:{gene.start}-{gene.end}", 
                min_total=params['min_total'], 
                min_alt_fraction=params['min_alt_fraction'],
                samples=sample_list
            )
            if events is not None and len(events) > 0:
                all_events.append(events)
                genes_with_events += 1
        except Exception:
            problematic_genes.append(gene.id)
    
    # Combine events
    if all_events:
        splice_events = pd.concat(all_events, ignore_index=True)
        results_pooled[group_name] = splice_events
        problematic_genes_pooled[group_name] = problematic_genes
        print(f"  Found {len(splice_events)} events across {genes_with_events} genes")
        
        # Save pooled results
        output_dir = f'{outpath}/pooled/{group_name}'
        os.makedirs(output_dir, exist_ok=True)
        splice_events.to_csv(f'{output_dir}/{group_name}_splicing_events.csv', index=False)
        
        with open(f'{output_dir}/{group_name}_problematic_genes.txt', 'w') as f:
            for gene_id in problematic_genes:
                f.write(f"{gene_id}\n")
    else:
        print(f"  No events found for group {group_name}")
        results_pooled[group_name] = pd.DataFrame()


PART 2: ANALYZING POOLED GROUPS

Processing group: I5
Samples in group: ['I5F', 'I5M']
Parameters: min_total=16, min_alt_fraction=0.125
  Found 1848 problematic genes out of 15472
  Found 37942 events across 4789 genes

Processing group: AD
Samples in group: ['Fad', 'Mad']
Parameters: min_total=16, min_alt_fraction=0.125
  Found 1846 problematic genes out of 15472
  Found 34924 events across 4684 genes


## Generate summary tables

In [4]:
print("\n" + "="*80)
print("PART 3: CREATING JOINT SUMMARY")
print("="*80)

# Get all unique event types across all results
all_event_types = set()

for sample, df in results_individual.items():
    if not df.empty:
        all_event_types.update(df['splice_type'].unique())

for group, df in results_pooled.items():
    if not df.empty:
        all_event_types.update(df['splice_type'].unique())

all_event_types = sorted(list(all_event_types))

# Build joint summary dataframe
joint_summary = []

# Add individual samples (sample column = sample name, type = 'individual')
for sample in individual_samples:
    row = {
        'sample': sample,
        'type': 'individual',
        'total_events': 0
    }
    
    if sample in results_individual and not results_individual[sample].empty:
        counts = results_individual[sample]['splice_type'].value_counts()
        row['total_events'] = len(results_individual[sample])
        for event_type in all_event_types:
            row[event_type] = counts.get(event_type, 0)
    else:
        for event_type in all_event_types:
            row[event_type] = 0
    
    joint_summary.append(row)

# Add pooled groups (sample column = group name, type = 'pooled')
for group_name in groups.keys():
    row = {
        'sample': group_name,
        'type': 'pooled',
        'total_events': 0
    }
    
    if group_name in results_pooled and not results_pooled[group_name].empty:
        counts = results_pooled[group_name]['splice_type'].value_counts()
        row['total_events'] = len(results_pooled[group_name])
        for event_type in all_event_types:
            row[event_type] = counts.get(event_type, 0)
    else:
        for event_type in all_event_types:
            row[event_type] = 0
    
    joint_summary.append(row)

# Create DataFrame
joint_df = pd.DataFrame(joint_summary)

# Reorder columns: sample first, then type, total_events, then event types
columns = ['sample', 'type', 'total_events'] + all_event_types
joint_df = joint_df[columns]

# Save joint summary
joint_csv = f'{outpath}/joint_event_summary.csv'
joint_df.to_csv(joint_csv, index=False)
print(f"\nJoint summary saved to: {joint_csv}")

# Create long format for R (sample column preserved)
long_format = []
for _, row in joint_df.iterrows():
    for event_type in all_event_types:
        long_format.append({
            'sample': row['sample'],
            'type': row['type'],
            'splice_type': event_type,
            'count': row[event_type]
        })

long_df = pd.DataFrame(long_format)
long_csv = f'{outpath}/joint_event_summary_long.csv'
long_df.to_csv(long_csv, index=False)
print(f"Long format summary saved to: {long_csv}")


PART 3: CREATING JOINT SUMMARY

Joint summary saved to: /home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/01.AS/joint_event_summary.csv
Long format summary saved to: /home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/01.AS/joint_event_summary_long.csv


## Print summary tables

In [5]:
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print("\nIndividual Samples:")
print("-" * 40)
for sample in individual_samples:
    if sample in results_individual and not results_individual[sample].empty:
        print(f"  {sample}: {len(results_individual[sample])} events")
    else:
        print(f"  {sample}: No events")

print("\nPooled Groups:")
print("-" * 40)
for group_name in groups.keys():
    if group_name in results_pooled and not results_pooled[group_name].empty:
        print(f"  {group_name}: {len(results_pooled[group_name])} events")
    else:
        print(f"  {group_name}: No events")

print("\n" + "="*80)
print("JOINT SUMMARY TABLE (first 5 rows)")
print("="*80)
print(joint_df.head().to_string(index=False))

print("\n" + "="*80)
print("To load in R:")
print("="*80)
print(f"  df <- read.csv('{joint_csv}')")
print(f"  df_long <- read.csv('{long_csv}')")


FINAL SUMMARY

Individual Samples:
----------------------------------------
  I1: 32704 events
  I2: 33797 events
  I3: 30385 events
  I4: 27719 events
  I5F: 22734 events
  I5M: 27461 events
  Fad: 20701 events
  Mad: 25792 events

Pooled Groups:
----------------------------------------
  I5: 37942 events
  AD: 34924 events

JOINT SUMMARY TABLE (first 5 rows)
sample       type  total_events  3AS  5AS  ES   IR  ME   PAS   TSS
    I1 individual         32704  747  967 816 2334 213 14829 12798
    I2 individual         33797  778 1037 835 2381 231 15182 13353
    I3 individual         30385  634  866 677 1947 169 12130 13962
    I4 individual         27719  576  845 752 1793 154 10620 12979
   I5F individual         22734  523  751 634 1335 133  9890  9468

To load in R:
  df <- read.csv('/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/01.AS/joint_event_summary.csv')
  df_long <- read.csv('/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.outpu